In [1]:
# TAU 6000 development AST extraction.
# Uses the fixed 16 kHz, 3-second, Top-8 AST setup with checkpoint/resume support.

from pathlib import Path
from contextlib import nullcontext
import time
import json
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import ASTFeatureExtractor, ASTForAudioClassification
warnings.filterwarnings('ignore')

# Project settings
PROJECT_ROOT = Path('C:\\Users\\22313906t\\Desktop\\TAU_project')
N_PER_CLASS = 600
EXPECTED_TOTAL = 6000
TAU_CLASSES = ['airport', 'bus', 'metro', 'metro station', 'park', 'public square', 'shopping mall', 'pedestrian street', 'street traffic', 'tram']

# Find development manifest
MANIFEST_PATH = PROJECT_ROOT / 'manifests' / 'tau_balanced_6000_experiment_manifest.csv'
if not MANIFEST_PATH.exists():
    matches = list(PROJECT_ROOT.rglob('tau_balanced_6000_experiment_manifest.csv'))
    if len(matches) == 1:
        MANIFEST_PATH = matches[0]
    elif len(matches) == 0:
        print('Possible development manifests:')
        for p in PROJECT_ROOT.rglob('*6000*manifest*.csv'):
            print(p)
        raise FileNotFoundError('Cannot find tau_balanced_6000_experiment_manifest.csv')
    else:
        print('Multiple 6000 manifests found:')
        for p in matches:
            print(p)
        raise RuntimeError('Choose the correct manifest manually.')

# Output paths
AST_RESULTS_DIR = PROJECT_ROOT / 'results_AST_6000'
AST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Aggregated recording-level AST events
AST_EVENTS_PATH = AST_RESULTS_DIR / 'tau_all_ast_events.csv'

# Top-8 events for each 3-second chunk
AST_SEGMENT_PATH = AST_RESULTS_DIR / 'tau_all_ast_segment_top8.csv'

# Strongest events per recording
AST_TOP_EVENTS_PATH = AST_RESULTS_DIR / 'tau_all_ast_top_events.csv'
AST_ERRORS_PATH = AST_RESULTS_DIR / 'tau_all_ast_errors.csv'
AST_SUMMARY_PATH = AST_RESULTS_DIR / 'tau_all_ast_summary.csv'
AST_CONFIG_PATH = AST_RESULTS_DIR / 'tau_all_ast_config.json'

# AST configuration
AST_MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
SAMPLE_RATE = 16000
CHUNK_SECONDS = 3.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_SECONDS)
TOP_K = 8
CHECKPOINT_EVERY = 50

# Load and validate the manifest
print('=' * 75)
print('TAU 6000 DEVELOPMENT — AST')
print('=' * 75)
print('\nManifest:')
print(MANIFEST_PATH)
df = pd.read_csv(MANIFEST_PATH)
required = {'clip_id', 'ground_truth', 'audio_path'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing manifest columns: {missing}')
df['clip_id'] = df['clip_id'].astype(str)
df['ground_truth'] = df['ground_truth'].astype(str)
print('\nRows:', len(df))
print('Unique IDs:', df['clip_id'].nunique())
class_counts = df['ground_truth'].value_counts().reindex(TAU_CLASSES, fill_value=0)
print('\nClass distribution:')
print(class_counts)
if len(df) != EXPECTED_TOTAL:
    raise ValueError(f'Expected {EXPECTED_TOTAL} clips, found {len(df)}.')
if df['clip_id'].nunique() != EXPECTED_TOTAL:
    raise ValueError('Expected 6000 unique clip IDs.')
if not (class_counts == N_PER_CLASS).all():
    raise ValueError('Test set is not balanced at 350 clips per category.')

# Verify audio files
audio_exists = df['audio_path'].apply(lambda p: Path(p).exists())
missing_audio = int((~audio_exists).sum())
print('\nMissing audio files:', missing_audio)
if missing_audio > 0:
    print(df.loc[~audio_exists, ['clip_id', 'audio_path']].head(30))
    raise FileNotFoundError(f'{missing_audio} audio files are missing.')
print('All 6000 audio files verified.')

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('\n' + '=' * 75)
print('DEVICE')
print('=' * 75)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Using:', DEVICE)

# Save config
config = {'experiment': 'development_6000', 'model': AST_MODEL_NAME, 'sample_rate': SAMPLE_RATE, 'chunk_seconds': CHUNK_SECONDS, 'top_k_per_chunk': TOP_K, 'activation': 'sigmoid', 'aggregation': ['max_score', 'mean_score', 'count', 'num_chunks', 'persistence'], 'manifest': str(MANIFEST_PATH), 'device': DEVICE}
with open(AST_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

# Load AST results
print('\n' + '=' * 75)
print('LOADING AST')
print('=' * 75)
print('Model:', AST_MODEL_NAME)
feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)
ast_model = ASTForAudioClassification.from_pretrained(AST_MODEL_NAME)
ast_model = ast_model.to(DEVICE)
ast_model.eval()
id2label = {int(k): v for k, v in ast_model.config.id2label.items()}
print('AudioSet labels:', len(id2label))

# Audio loading
def load_audio(audio_path):
    waveform, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    waveform = np.asarray(waveform, dtype=np.float32)
    if len(waveform) == 0:
        raise RuntimeError(f'Empty audio file: {audio_path}')
    return waveform

# Split into 3-second chunks
def split_audio(waveform):
    chunks = []
    for start in range(0, len(waveform), CHUNK_SAMPLES):
        chunk = waveform[start:start + CHUNK_SAMPLES]
        if len(chunk) == 0:
            continue
        if len(chunk) < CHUNK_SAMPLES:
            padded = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
            padded[:len(chunk)] = chunk
            chunk = padded
        chunks.append(chunk.astype(np.float32))
    if len(chunks) == 0:
        raise RuntimeError('No valid audio chunks created.')
    return chunks

# Run AST on all chunks of one recording
def classify_chunks(chunks):
    inputs = feature_extractor(chunks, sampling_rate=SAMPLE_RATE, return_tensors='pt')
    input_values = inputs['input_values'].to(DEVICE)
    if DEVICE == 'cuda':
        precision_context = torch.autocast(device_type='cuda', dtype=torch.float16)
    else:
        precision_context = nullcontext()
    with torch.inference_mode():
        with precision_context:
            outputs = ast_model(input_values=input_values)
            logits = outputs.logits
            scores = torch.sigmoid(logits)
    scores = scores.float().cpu().numpy()
    return scores

# Extract top-8 events / chunk
def process_one_audio(clip_id, ground_truth, audio_path):
    waveform = load_audio(audio_path)
    chunks = split_audio(waveform)
    scores = classify_chunks(chunks)
    num_chunks = len(chunks)
    segment_rows = []
    for chunk_index in range(num_chunks):
        chunk_scores = scores[chunk_index]
        top_indices = np.argsort(chunk_scores)[::-1][:TOP_K]
        for rank, label_id in enumerate(top_indices, start=1):
            segment_rows.append({'clip_id': clip_id, 'ground_truth': ground_truth, 'chunk_index': chunk_index, 'rank': rank, 'label_id': int(label_id), 'label': id2label[int(label_id)], 'score': float(chunk_scores[label_id]), 'num_chunks': num_chunks})
    segment_df = pd.DataFrame(segment_rows)
    aggregated = segment_df.groupby(['clip_id', 'ground_truth', 'label'], as_index=False).agg(max_score=('score', 'max'), mean_score=('score', 'mean'), count=('chunk_index', 'nunique'))
    aggregated['num_chunks'] = num_chunks
    aggregated['persistence'] = aggregated['count'] / aggregated['num_chunks']
    aggregated = aggregated.sort_values(['max_score', 'persistence'], ascending=[False, False]).reset_index(drop=True)
    return (aggregated, segment_df)

# One-audio test
print('\n' + '=' * 75)
print('ONE-AUDIO TEST')
print('=' * 75)
test_row = df.iloc[0]
print('Clip:', test_row['clip_id'])
print('Ground truth:', test_row['ground_truth'])
test_events, test_segments = process_one_audio(clip_id=str(test_row['clip_id']), ground_truth=str(test_row['ground_truth']), audio_path=str(test_row['audio_path']))
print('\nNumber of chunks:', int(test_events['num_chunks'].iloc[0]))
print('\nTop aggregated events:')
print(test_events[['label', 'max_score', 'count', 'num_chunks', 'persistence']].head(15).to_string(index=False))
print('\nOne-audio AST test successful.')

# Automatic resume
print('\n' + '=' * 75)
print('CHECKPOINT / RESUME')
print('=' * 75)
experiment_ids = set(df['clip_id'].astype(str))
if AST_EVENTS_PATH.exists():
    existing_events = pd.read_csv(AST_EVENTS_PATH)
    existing_events['clip_id'] = existing_events['clip_id'].astype(str)
    existing_events = existing_events[existing_events['clip_id'].isin(experiment_ids)].copy()
    processed_ids = set(existing_events['clip_id'])
else:
    existing_events = pd.DataFrame()
    processed_ids = set()
if AST_SEGMENT_PATH.exists():
    existing_segments = pd.read_csv(AST_SEGMENT_PATH)
else:
    existing_segments = pd.DataFrame()
if AST_ERRORS_PATH.exists():
    existing_errors = pd.read_csv(AST_ERRORS_PATH)
else:
    existing_errors = pd.DataFrame()
remaining_df = df[~df['clip_id'].isin(processed_ids)].copy().reset_index(drop=True)
print('Already processed:', len(processed_ids))
print('Remaining:', len(remaining_df))

# Buffers
event_buffers = []
segment_buffers = []
error_buffers = []

# Save checkpoint
def save_checkpoint():
    global existing_events
    global existing_segments
    global existing_errors
    global event_buffers
    global segment_buffers
    global error_buffers
    if event_buffers:
        new_events = pd.concat(event_buffers, ignore_index=True)
        existing_events = pd.concat([existing_events, new_events], ignore_index=True)
        existing_events['clip_id'] = existing_events['clip_id'].astype(str)
        existing_events = existing_events.drop_duplicates(subset=['clip_id', 'label'], keep='last')
        existing_events.to_csv(AST_EVENTS_PATH, index=False)
        event_buffers.clear()
    if segment_buffers:
        new_segments = pd.concat(segment_buffers, ignore_index=True)
        existing_segments = pd.concat([existing_segments, new_segments], ignore_index=True)
        existing_segments['clip_id'] = existing_segments['clip_id'].astype(str)
        existing_segments = existing_segments.drop_duplicates(subset=['clip_id', 'chunk_index', 'label'], keep='last')
        existing_segments.to_csv(AST_SEGMENT_PATH, index=False)
        segment_buffers.clear()
    if error_buffers:
        new_errors = pd.DataFrame(error_buffers)
        existing_errors = pd.concat([existing_errors, new_errors], ignore_index=True)
        existing_errors = existing_errors.drop_duplicates(subset=['clip_id'], keep='last')
        existing_errors.to_csv(AST_ERRORS_PATH, index=False)
        error_buffers.clear()

# Full AST run
print('\n' + '=' * 75)
print('STARTING FULL AST DEVELOPMENT RUN')
print('=' * 75)
print('Audio clips to process:', len(remaining_df))
print('Chunk duration:', CHUNK_SECONDS, 'seconds')
print('Top-K / chunk:', TOP_K)
start_time = time.time()
since_checkpoint = 0
for idx, row in remaining_df.iterrows():
    clip_id = str(row['clip_id'])
    ground_truth = str(row['ground_truth'])
    audio_path = str(row['audio_path'])
    try:
        events_df, segments_df = process_one_audio(clip_id=clip_id, ground_truth=ground_truth, audio_path=audio_path)
        event_buffers.append(events_df)
        segment_buffers.append(segments_df)
    except Exception as e:
        print('\nERROR:', clip_id, str(e))
        error_buffers.append({'clip_id': clip_id, 'ground_truth': ground_truth, 'audio_path': audio_path, 'error': str(e)})
    since_checkpoint += 1
    if (idx + 1) % 25 == 0 or idx == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed if elapsed > 0 else 0
        print(f'[{idx + 1:4d}/{len(remaining_df):4d}] {ground_truth:20s} | {clip_id} | {rate:.2f} clips/s')
    if since_checkpoint >= CHECKPOINT_EVERY:
        save_checkpoint()
        since_checkpoint = 0
        saved_ids = existing_events['clip_id'].nunique() if len(existing_events) > 0 else 0
        print(f'----- CHECKPOINT SAVED ({saved_ids} clips) -----')

# Final checkpoint
save_checkpoint()
elapsed_minutes = (time.time() - start_time) / 60
print('\n' + '=' * 75)
print('AST PROCESSING FINISHED')
print('=' * 75)
print('Elapsed minutes:', round(elapsed_minutes, 2))

# Load final AST events
ast_df = pd.read_csv(AST_EVENTS_PATH)
ast_df['clip_id'] = ast_df['clip_id'].astype(str)
ast_df = ast_df[ast_df['clip_id'].isin(experiment_ids)].copy()
processed_count = ast_df['clip_id'].nunique()
print('\nProcessed AST clips:', processed_count)

# Build top-event file
# Top 20 aggregated events per recording.
# Main ontology should still use AST_EVENTS_PATH.
TOP_EVENTS_PER_CLIP = 20
top_events_df = ast_df.sort_values(['clip_id', 'max_score', 'persistence'], ascending=[True, False, False]).groupby('clip_id', group_keys=False).head(TOP_EVENTS_PER_CLIP).copy()
top_events_df.to_csv(AST_TOP_EVENTS_PATH, index=False)

# Processing errors
if AST_ERRORS_PATH.exists():
    errors_df = pd.read_csv(AST_ERRORS_PATH)
else:
    errors_df = pd.DataFrame()
print('Processing errors:', len(errors_df))
if processed_count != EXPECTED_TOTAL:
    print(f'\nWARNING: AST results exist for {processed_count}/{EXPECTED_TOTAL} clips.')
    print('Rerun this cell to resume missing clips.')
else:
    print('\nAll 6000 development clips have AST results.')

# Summary of AST event extraction
clip_chunk_counts = ast_df.groupby('clip_id')['num_chunks'].max()
summary_df = pd.DataFrame([{'experiment': 'development_6000', 'model': AST_MODEL_NAME, 'number_of_manifest_clips': len(df), 'number_of_processed_clips': processed_count, 'processing_errors': len(errors_df), 'sample_rate': SAMPLE_RATE, 'chunk_seconds': CHUNK_SECONDS, 'top_k_per_chunk': TOP_K, 'number_of_aggregated_event_rows': len(ast_df), 'mean_chunks_per_clip': float(clip_chunk_counts.mean()), 'min_chunks_per_clip': int(clip_chunk_counts.min()), 'max_chunks_per_clip': int(clip_chunk_counts.max())}])
summary_df.to_csv(AST_SUMMARY_PATH, index=False)
print('\n' + '=' * 75)
print('AST SUMMARY')
print('=' * 75)
print(summary_df.to_string(index=False))

# Most common events
event_occurrence = ast_df.groupby('label')['clip_id'].nunique().sort_values(ascending=False)
print('\n' + '=' * 75)
print('MOST COMMON AST EVENTS')
print('=' * 75)
print(event_occurrence.head(30).to_string())

# Show example
example_id = df.iloc[0]['clip_id']
example = ast_df[ast_df['clip_id'] == str(example_id)].sort_values('max_score', ascending=False)
print('\n' + '=' * 75)
print('EXAMPLE AGGREGATED EVENTS')
print('=' * 75)
print(example[['label', 'max_score', 'mean_score', 'count', 'num_chunks', 'persistence']].head(15).to_string(index=False))

# Files saved
print('\n' + '=' * 75)
print('FILES SAVED')
print('=' * 75)
print('\nMain aggregated AST events:')
print(AST_EVENTS_PATH)
print('\nSegment-level Top-8:')
print(AST_SEGMENT_PATH)
print('\nTop aggregated events:')
print(AST_TOP_EVENTS_PATH)
print('\nSummary:')
print(AST_SUMMARY_PATH)
print('\nErrors:')
print(AST_ERRORS_PATH)
print('\nConfig:')
print(AST_CONFIG_PATH)
print('\n' + '=' * 75)
print('DONE')
print('=' * 75)
print('\nNext stage:')
print('Combine:')
print(PROJECT_ROOT / 'results_CLIP' / 'tau_6000_clip_predictions.csv')
print('\nwith:')
print(AST_EVENTS_PATH)
print('\nand apply the frozen ontology.')


C:\Users\22313906t\AppData\Local\anaconda3\envs\tau_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TAU 6000 DEVELOPMENT — AST

Manifest:
C:\Users\22313906t\Desktop\TAU_project\manifests\tau_balanced_6000_experiment_manifest.csv

Rows: 6000
Unique IDs: 6000

Class distribution:
ground_truth
airport              600
bus                  600
metro                600
metro station        600
park                 600
public square        600
shopping mall        600
pedestrian street    600
street traffic       600
tram                 600
Name: count, dtype: int64

Missing audio files: 0
All 6000 audio files verified.

DEVICE
PyTorch: 2.13.0+cu126
CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation Laptop GPU
Using: cuda

LOADING AST
Model: MIT/ast-finetuned-audioset-10-10-0.4593


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 561.60it/s]


AudioSet labels: 527

ONE-AUDIO TEST
Clip: metro-lisbon-1055-41182
Ground truth: metro

Number of chunks: 4

Top aggregated events:
                     label  max_score  count  num_chunks  persistence
                    Speech   0.559570      4           4         1.00
                   Vehicle   0.300537      4           4         1.00
                     Train   0.147583      4           4         1.00
                     Music   0.110291      1           4         0.25
                       Car   0.102661      4           4         1.00
                       Bus   0.092529      4           4         1.00
              Sliding door   0.056854      1           4         0.25
 Railroad car, train wagon   0.054108      1           4         0.25
Subway, metro, underground   0.053406      1           4         0.25
                   Silence   0.046204      1           4         0.25
            Rail transport   0.045013      1           4         0.25
            Car passing by  